# Day 5 — ETL Notebook: clean and load to MySQL

Objective: Build a small ETL inside this notebook. Functions: load_raw, clean_df, write_replace, write_upsert, main.

This notebook is safe ASCII-only to avoid JSON parse errors. Run cells top-to-bottom.

In [ ]:
# Imports and basic helpers
import os
import json
import hashlib
from pathlib import Path
import logging
import pandas as pd
from sqlalchemy import create_engine

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

def load_raw(path_json='raw_data.json', path_csv='raw_data.csv'):
    p_json = Path(path_json)
    p_csv = Path(path_csv)
    if p_json.exists():
        logging.info('Loading raw JSON: %s', p_json)
        with p_json.open('r', encoding='utf-8') as f:
            raw = json.load(f)
        return pd.DataFrame(raw)
    if p_csv.exists():
        logging.info('Loading raw CSV: %s', p_csv)
        return pd.read_csv(p_csv, parse_dates=['scraped_at'], encoding='utf-8')
    # sample fallback
    logging.info('No raw data found - creating sample')
    sample = [
        {'scraped_at': '2026-06-30T12:00:00', 'item': 'Croissant', 'price_text': 'CHF 2.50', 'price': None, 'quantity': None, 'store': 'Main', 'url': 'https://example.com/croissant'},
        {'scraped_at': '2026-06-30T12:05:00', 'item': 'Baguette', 'price_text': 'CHF 3.00', 'price': None, 'quantity': 2, 'store': 'Main', 'url': 'https://example.com/baguette'},
    ]
    return pd.DataFrame(sample)


In [ ]:
# Unique key helper
def make_unique_key_row(row):
    url = str(row.get('url', ''))
    item = str(row.get('item', '')).strip().lower()
    scraped = row.get('scraped_at')
    try:
        scraped_date = pd.to_datetime(scraped, errors='coerce').date().isoformat()
    except Exception:
        scraped_date = str(scraped)
    s = '|'.join([url, item, scraped_date])
    return hashlib.sha256(s.encode('utf-8')).hexdigest()
